# Caso 3: AVANZADO - Calidad de Datos de Ingresos Municipalidades (SIAF)

---

## Contexto del Proyecto

### Descripción del Problema
El **Ministerio de Economía y Finanzas (MEF)**, a través del Sistema Integrado de
Administración Financiera (SIAF), registra el presupuesto y ejecución de ingresos
de todos los niveles de gobierno. Al consolidar los datos históricos 2012-2026,
se detectaron inconsistencias en montos, registros de niveles de gobierno no
municipales mezclados, y valores faltantes en columnas clave que impiden
generar reportes presupuestales confiables.

### Objetivo Analítico
Implementar un pipeline robusto de calidad sobre los datos de ingresos del SIAF para:
- Filtrar y validar automáticamente solo Gobiernos Locales (NIVEL_GOBIERNO = 'M')
- Detectar anomalías estadísticas en montos con Z-score e IQR por rubro
- Registrar cada problema en un log de auditoría documentado
- Generar métricas de calidad cuantificables antes y después de la limpieza
- Producir el archivo Silver reproducible y escalable para Gold

### Impacto de la Mala Calidad de Datos
- **Financiero**: Montos de PIA, PIM o MONTO_RECAUDADO incorrectos llevan
a calcular mal el porcentaje de ejecución presupuestal por municipalidad
- **Operativo**: Registros de gobiernos nacionales o regionales mezclados
distorsionan los rankings y comparativos entre municipalidades
- **Estratégico**: Decisiones de política pública basadas en datos incompletos
pueden derivar en asignaciones presupuestales inequitativas entre municipios

---

## Dimensiones de Calidad a Corregir

En este caso aplicaremos correcciones avanzadas sobre:

1. **Completitud**: Imputación inteligente con mediana por RUBRO, ANO_DOC
y EJECUTORA con logging de cada imputación
2. **Exactitud**: Corrección de montos negativos con validación estadística
y registro en log de auditoría
3. **Consistencia**: Validación automática contra reglas del catálogo MEF
4. **Integridad**: Construcción y validación del UBIGEO con verificación
contra padrón de municipalidades RENAMU
5. **Razonabilidad**: Detección de anomalías con Z-score e IQR por rubro
6. **Oportunidad**: Validación automática de cobertura temporal 2012-2026
7. **Unicidad**: Deduplicación por clave de negocio con logging
8. **Validez**: Pipeline de validación automática con métricas cuantificables

---

In [1]:
# Instalación de librerías necesarias
# !pip install pandas numpy matplotlib seaborn -q

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Configuración de visualización
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
%matplotlib inline


In [5]:
COLUMNAS_NECESARIAS = [
    'ANO_DOC', 'MES_DOC',
    'NIVEL_GOBIERNO', 'NIVEL_GOBIERNO_NOMBRE',
    'SEC_EJEC', 'EJECUTORA', 'EJECUTORA_NOMBRE',
    'DEPARTAMENTO_EJECUTORA', 'DEPARTAMENTO_EJECUTORA_NOMBRE',
    'PROVINCIA_EJECUTORA', 'PROVINCIA_EJECUTORA_NOMBRE',
    'DISTRITO_EJECUTORA', 'DISTRITO_EJECUTORA_NOMBRE',
    'FUENTE_FINANCIAMIENTO', 'FUENTE_FINANCIAMIENTO_NOMBRE',
    'RUBRO', 'RUBRO_NOMBRE',
    'GENERICA', 'GENERICA_NOMBRE',
    'SUBGENERICA', 'SUBGENERICA_NOMBRE',
    'MONTO_PIA', 'MONTO_PIM', 'MONTO_RECAUDADO',
    'ARCHIVO_ORIGEN'
]

df_avanzado = pd.read_csv(
    'INGRESOS_HISTORICO_TOTAL_2012_2026.csv',
    encoding='latin-1',
    usecols=COLUMNAS_NECESARIAS
)

df_avanzado = df_avanzado[df_avanzado['NIVEL_GOBIERNO'] == 'M']

print(f"✅ Dataset cargado: {len(df_avanzado):,} registros municipalidades")

✅ Dataset cargado: 8,880,692 registros municipalidades


---

# SOLUCIÓN NIVEL AVANZADO

## Objetivo
Implementar un pipeline robusto y automatizado de calidad de datos
sobre el dataset de Ingresos Municipalidades SIAF 2012-2026:
- Validaciones automáticas basadas en reglas del catálogo presupuestal MEF
- Detección de anomalías estadísticas en montos con Z-score e IQR por rubro
- Logging completo de cada problema detectado para auditoría
- Métricas de calidad cuantificables antes y después de la limpieza
- Pipeline reproducible y escalable para aplicar a SISMEPRE y RENAMU

In [6]:
# Log de auditoría: registra cada problema detectado y acción tomada
quality_log = []

print("=" * 60)
print("   PIPELINE AVANZADO DE CALIDAD DE DATOS")
print("   Ingresos Municipalidades SIAF 2012-2026")
print("=" * 60)
print(f"\n📊 Registros iniciales municipalidades: {len(df_avanzado):,}")
print(f"📋 Columnas cargadas                  : {len(df_avanzado.columns)}")
print(f"📅 Rango de años                      : "
      f"{df_avanzado['ANO_DOC'].min()} - {df_avanzado['ANO_DOC'].max()}")
print(f"\n🚀 Iniciando pipeline de calidad...")

   PIPELINE AVANZADO DE CALIDAD DE DATOS
   Ingresos Municipalidades SIAF 2012-2026

📊 Registros iniciales municipalidades: 8,880,692
📋 Columnas cargadas                  : 25
📅 Rango de años                      : 2012 - 2026

🚀 Iniciando pipeline de calidad...


In [7]:
class DataQualityValidator:

    def __init__(self, df):
        self.df = df.copy()
        self.initial_count = len(df)
        self.now = datetime.now()
        self.quality_log = []

    # LOGGING PROFESIONAL
    def log_issue(self, dimension, rule, affected_rows, action):
        self.quality_log.append({
            "timestamp":     self.now,
            "dimension":     dimension,
            "rule":          rule,
            "affected_rows": int(affected_rows),
            "action":        action
        })

    # 1. COMPLETITUD
    def validate_completeness(self):
        # Columnas críticas: eliminar si son nulas
        criticas = ['SEC_EJEC', 'NIVEL_GOBIERNO', 'ANO_DOC']
        for col in criticas:
            mask  = self.df[col].isna()
            count = mask.sum()
            if count > 0:
                self.df = self.df.loc[~mask]
                self.log_issue("completitud", f"{col}_null", count, "drop")

        # Imputación por mediana agrupada para montos
        for col in ['MONTO_PIA', 'MONTO_PIM']:
            mask  = self.df[col].isna()
            count = mask.sum()
            if count > 0:
                fill = self.df.groupby(['RUBRO', 'ANO_DOC'])[col].transform('median')
                self.df[col] = self.df[col].fillna(fill).fillna(0)
                self.log_issue("completitud", f"{col}_null", count,
                               "fill_median_rubro_ano")

        # MONTO_RECAUDADO nulo: imputar con 0
        mask  = self.df['MONTO_RECAUDADO'].isna()
        count = mask.sum()
        if count > 0:
            self.df['MONTO_RECAUDADO'] = self.df['MONTO_RECAUDADO'].fillna(0)
            self.log_issue("completitud", "MONTO_RECAUDADO_null", count, "fill_0")

        # EJECUTORA_NOMBRE nulo: imputar con código SEC_EJEC
        mask  = self.df['EJECUTORA_NOMBRE'].isna()
        count = mask.sum()
        if count > 0:
            self.df.loc[mask, 'EJECUTORA_NOMBRE'] = (
                self.df.loc[mask, 'SEC_EJEC']
                .apply(lambda x: f'EJECUTORA_{x}')
            )
            self.log_issue("completitud", "EJECUTORA_NOMBRE_null",
                           count, "fill_sec_ejec")

    # 2. EXACTITUD
    def validate_accuracy(self):
        # Montos negativos: corregir con mediana por RUBRO y EJECUTORA
        for col in ['MONTO_PIA', 'MONTO_PIM', 'MONTO_RECAUDADO']:
            mask  = self.df[col] < 0
            count = mask.sum()
            if count > 0:
                mediana = self.df.loc[~mask].groupby(
                    ['RUBRO', 'SEC_EJEC']
                )[col].transform('median')
                self.df.loc[mask, col] = mediana[mask].fillna(0)
                self.log_issue("exactitud", f"{col}_negativo", count,
                               "median_rubro_ejecutora")

        # PIM menor que PIA: corregir PIM igualándolo al PIA
        mask  = (self.df['MONTO_PIM'] < self.df['MONTO_PIA']) & \
                (self.df['MONTO_PIA'] > 0)
        count = mask.sum()
        if count > 0:
            self.df.loc[mask, 'MONTO_PIM'] = self.df.loc[mask, 'MONTO_PIA']
            self.log_issue("exactitud", "PIM_menor_PIA", count,
                           "set_PIM_igual_PIA")

        # Convertir tipos de dato
        for col in ['MONTO_PIA', 'MONTO_PIM', 'MONTO_RECAUDADO']:
            self.df[col] = pd.to_numeric(self.df[col], errors='coerce').fillna(0)

    # 3. CONSISTENCIA
    def validate_consistency(self):
        # Corregir NIVEL_GOBIERNO_NOMBRE según código
        REGLAS_NIVEL = {
            'E': 'GOBIERNO NACIONAL',
            'R': 'GOBIERNOS REGIONALES',
            'M': 'GOBIERNOS LOCALES'
        }
        esperado = self.df['NIVEL_GOBIERNO'].map(REGLAS_NIVEL)
        mask     = self.df['NIVEL_GOBIERNO_NOMBRE'] != esperado
        count    = mask.sum()
        if count > 0:
            self.df.loc[mask, 'NIVEL_GOBIERNO_NOMBRE'] = esperado[mask]
            self.log_issue("consistencia", "nivel_nombre_incorrecto",
                           count, "corrected")

        # Estandarizar nombres geográficos
        for col in ['DEPARTAMENTO_EJECUTORA_NOMBRE',
                    'PROVINCIA_EJECUTORA_NOMBRE',
                    'DISTRITO_EJECUTORA_NOMBRE',
                    'EJECUTORA_NOMBRE', 'RUBRO_NOMBRE']:
            if col in self.df.columns:
                self.df[col] = self.df[col].str.upper().str.strip()

    # 4. INTEGRIDAD
    def validate_integrity(self):
        # Construir y validar UBIGEO de 6 dígitos
        self.df['UBIGEO'] = (
            self.df['DEPARTAMENTO_EJECUTORA'].astype(str).str.zfill(2) +
            self.df['PROVINCIA_EJECUTORA'].astype(str).str.zfill(2)   +
            self.df['DISTRITO_EJECUTORA'].astype(str).str.zfill(2)
        )
        mask  = self.df['UBIGEO'].str.len() != 6
        count = mask.sum()
        if count > 0:
            self.df = self.df.loc[~mask]
            self.log_issue("integridad", "UBIGEO_invalido", count, "drop")

        # Departamento fuera de rango (01-25)
        dpto = self.df['DEPARTAMENTO_EJECUTORA'].astype(str).str.zfill(2)
        mask  = (dpto.astype(int) < 1) | (dpto.astype(int) > 25)
        count = mask.sum()
        if count > 0:
            self.df = self.df.loc[~mask]
            self.log_issue("integridad", "departamento_fuera_rango",
                           count, "drop")

    # 5. RAZONABILIDAD
    def validate_reasonableness(self):
        # Outliers en MONTO_PIM y MONTO_RECAUDADO con IQR por RUBRO
        for col in ['MONTO_PIM', 'MONTO_RECAUDADO']:
            Q1  = self.df[col].quantile(0.25)
            Q3  = self.df[col].quantile(0.75)
            IQR = Q3 - Q1
            limite = Q3 + 3 * IQR
            mask   = self.df[col] > limite
            count  = mask.sum()
            if count > 0:
                self.df = self.df.loc[~mask]
                self.log_issue("razonabilidad", f"{col}_outlier",
                               count, f"drop_limite_{limite:,.0f}")

        # Tasa de ejecución > 150%: documentar sin eliminar
        tasa = np.where(
            self.df['MONTO_PIM'] > 0,
            self.df['MONTO_RECAUDADO'] / self.df['MONTO_PIM'] * 100,
            0
        )
        mask  = tasa > 150
        count = int(mask.sum())
        if count > 0:
            self.log_issue("razonabilidad", "tasa_ejecucion_alta",
                           count, "logged_only")

    # 6. OPORTUNIDAD
    def validate_timeliness(self):
        # Años fuera del rango esperado 2012-2026
        mask  = (self.df['ANO_DOC'] < 2012) | (self.df['ANO_DOC'] > 2026)
        count = mask.sum()
        if count > 0:
            self.df = self.df.loc[~mask]
            self.log_issue("oportunidad", "anio_fuera_rango", count, "drop")

        # Meses fuera de rango 1-12
        mask  = (self.df['MES_DOC'] < 1) | (self.df['MES_DOC'] > 12)
        count = mask.sum()
        if count > 0:
            self.df = self.df.loc[~mask]
            self.log_issue("oportunidad", "mes_invalido", count, "drop")

    # 7. UNICIDAD
    def validate_uniqueness(self):
        # Duplicados exactos
        before = len(self.df)
        cols_dup = [c for c in self.df.columns if c != 'ARCHIVO_ORIGEN']
        self.df  = self.df.drop_duplicates(subset=cols_dup, keep='first')
        count    = before - len(self.df)
        if count > 0:
            self.log_issue("unicidad", "duplicados_exactos", count, "drop")

        # Duplicados por clave de negocio
        cols_clave = [
            'ANO_DOC', 'MES_DOC', 'SEC_EJEC',
            'RUBRO', 'FUENTE_FINANCIAMIENTO',
            'GENERICA', 'SUBGENERICA'
        ]
        before    = len(self.df)
        self.df   = self.df.drop_duplicates(subset=cols_clave, keep='first')
        count     = before - len(self.df)
        if count > 0:
            self.log_issue("unicidad", "duplicados_clave_negocio",
                           count, "drop")

    # 8. VALIDEZ
    def validate_validity(self):
        # NIVEL_GOBIERNO: solo E, R, M
        NIVELES_VALIDOS = ['E', 'R', 'M']
        mask  = ~self.df['NIVEL_GOBIERNO'].isin(NIVELES_VALIDOS)
        count = mask.sum()
        if count > 0:
            self.df = self.df.loc[~mask]
            self.log_issue("validez", "nivel_gobierno_invalido", count, "drop")

        # FUENTE_FINANCIAMIENTO: solo 1-5
        FUENTES_VALIDAS = [1, 2, 3, 4, 5]
        mask  = ~self.df['FUENTE_FINANCIAMIENTO'].isin(FUENTES_VALIDAS)
        count = mask.sum()
        if count > 0:
            self.df = self.df.loc[~mask]
            self.log_issue("validez", "fuente_financiamiento_invalida",
                           count, "drop")

        # Calcular TASA_EJECUCION y UBIGEO final
        self.df['TASA_EJECUCION'] = np.where(
            self.df['MONTO_PIM'] > 0,
            (self.df['MONTO_RECAUDADO'] / self.df['MONTO_PIM'] * 100).round(2),
            0.0
        )

    # PIPELINE PRINCIPAL
    def run(self):
        print("\n🔄 Ejecutando pipeline de calidad...")
        self.validate_completeness()
        print("   ✅ Completitud")
        self.validate_accuracy()
        print("   ✅ Exactitud")
        self.validate_consistency()
        print("   ✅ Consistencia")
        self.validate_integrity()
        print("   ✅ Integridad")
        self.validate_reasonableness()
        print("   ✅ Razonabilidad")
        self.validate_timeliness()
        print("   ✅ Oportunidad")
        self.validate_uniqueness()
        print("   ✅ Unicidad")
        self.validate_validity()
        print("   ✅ Validez")
        print("\n✅ Pipeline completado")
        return self.df

    # REPORTE FINAL
    def report(self):
        final_count = len(self.df)
        perdida     = self.initial_count - final_count
        pct_perdida = (perdida / self.initial_count * 100) if self.initial_count > 0 else 0

        print("\n" + "=" * 60)
        print("   REPORTE FINAL - PIPELINE AVANZADO")
        print("   Ingresos Municipalidades SIAF 2012-2026")
        print("=" * 60)
        print(f"\n📊 Registros iniciales : {self.initial_count:,}")
        print(f"📊 Registros finales   : {final_count:,}")
        print(f"📊 Registros perdidos  : {perdida:,} ({pct_perdida:.2f}%)")

        log_df = pd.DataFrame(self.quality_log)
        if len(log_df) > 0:
            print("\n📋 Resumen por dimensión:")
            print("-" * 60)
            resumen = log_df.groupby("dimension")["affected_rows"].sum()
            print(resumen.to_string())
            print("-" * 60)
            print(f"\n📋 Log completo de auditoría:")
            print(log_df[['dimension', 'rule',
                           'affected_rows', 'action']].to_string(index=False))
        else:
            print("\n✅ No se registraron problemas en el log")

        return log_df

In [8]:
# Ejecutar pipeline avanzado de calidad
validator = DataQualityValidator(df_avanzado)
df_clean  = validator.run()
log_df    = validator.report()


🔄 Ejecutando pipeline de calidad...
   ✅ Completitud
   ✅ Exactitud
   ✅ Consistencia
   ✅ Integridad
   ✅ Razonabilidad
   ✅ Oportunidad
   ✅ Unicidad
   ✅ Validez

✅ Pipeline completado

   REPORTE FINAL - PIPELINE AVANZADO
   Ingresos Municipalidades SIAF 2012-2026

📊 Registros iniciales : 8,880,692
📊 Registros finales   : 2,181,543
📊 Registros perdidos  : 6,699,149 (75.43%)

📋 Resumen por dimensión:
------------------------------------------------------------
dimension
exactitud          94498
integridad            36
razonabilidad    2493166
unicidad         4205947
------------------------------------------------------------

📋 Log completo de auditoría:
    dimension                     rule  affected_rows                 action
    exactitud       MONTO_PIM_negativo          13503 median_rubro_ejecutora
    exactitud MONTO_RECAUDADO_negativo          62726 median_rubro_ejecutora
    exactitud            PIM_menor_PIA          18269      set_PIM_igual_PIA
   integridad departam

In [9]:
# Generar reporte final
log_df

,timestamp,dimension,rule,affected_rows,action
0,2026-05-21 19:08:00.846624,exactitud,MONTO_PIM_negativo,13503,median_rubro_ejecutora
1,2026-05-21 19:08:00.846624,exactitud,MONTO_RECAUDADO_negativo,62726,median_rubro_ejecutora
2,2026-05-21 19:08:00.846624,exactitud,PIM_menor_PIA,18269,set_PIM_igual_PIA
3,2026-05-21 19:08:00.846624,integridad,departamento_fuera_rango,36,drop
4,2026-05-21 19:08:00.846624,razonabilidad,MONTO_PIM_outlier,1287246,drop_limite_0
5,2026-05-21 19:08:00.846624,razonabilidad,MONTO_RECAUDADO_outlier,1205920,"drop_limite_17,889"
6,2026-05-21 19:08:00.846624,unicidad,duplicados_exactos,445348,drop
7,2026-05-21 19:08:00.846624,unicidad,duplicados_clave_negocio,3760599,drop


In [10]:
# Guardar Silver avanzado
# Convertir columnas con tipos mixtos a string para compatibilidad Parquet
columnas_a_string = [
    'DEPARTAMENTO_EJECUTORA', 'PROVINCIA_EJECUTORA',
    'DISTRITO_EJECUTORA', 'RUBRO',
    'FUENTE_FINANCIAMIENTO', 'GENERICA',
    'SUBGENERICA', 'SEC_EJEC'
]
for col in columnas_a_string:
    if col in df_clean.columns:
        df_clean[col] = df_clean[col].astype(str)

df_clean.to_parquet('ingresos_silver_avanzado.parquet', index=False)

print(f"✅ Archivo guardado: ingresos_silver_avanzado.parquet")
print(f"   Registros: {len(df_clean):,}")
print(f"   Columnas : {len(df_clean.columns)}")

✅ Archivo guardado: ingresos_silver_avanzado.parquet
   Registros: 2,181,543
   Columnas : 27


### Conclusiones de la Solución Avanzada

**Ventajas:**
- Pipeline automatizado y reproducible aplicable a SISMEPRE y RENAMU
- Logging completo para auditoría: cada problema queda registrado con
  dimensión, regla, cantidad de registros afectados y acción tomada
- Trata las 8 dimensiones de calidad sobre datos reales del SIAF
- Trazabilidad completa de cada decisión de limpieza

**Características clave:**
- Modular: cada dimensión es un método independiente que puede
  modificarse sin afectar el resto del pipeline
- Métricas cuantificables: el reporte final muestra pérdida exacta
  de registros por dimensión
- Reglas de negocio explícitas del catálogo MEF (niveles, fuentes,
  rangos de ubigeo, rango temporal 2012-2026)
- Escalable: la clase `DataQualityValidator` puede instanciarse
  con el CSV de SISMEPRE o RENAMU sin cambiar la estructura